# StochasticQuant-F90 — Convergence & Backtesting

> **All data in this notebook is synthetic / simulated** (seeded Monte Carlo, no real portfolio, no real market data). It exists to validate the numerics, not to describe any real exposure.

This notebook reproduces, interactively, what `scripts/convergence.py` and `scripts/backtest.py` do from the command line: (1) VaR/CVaR convergence in the number of scenarios, (2) Euler vs Milstein error for GBM, (3) Kupiec backtesting of a well-specified vs a heavy-tailed VaR model.

In [ ]:
import sys
sys.path.insert(0, '..')
sys.path.insert(0, '../src')
import numpy as np
from models import simulate_gbm, simulate_vasicek_losses, vasicek_var_analytic
from risk_metrics import value_at_risk, expected_shortfall, kupiec_test
from scipy.stats import norm
print('imports OK')

## 1. VaR/CVaR convergence — error shrinks like $1/\sqrt{N}$

Reference: the Vasicek large-homogeneous-portfolio limit, $\mathrm{VaR}_\alpha = N\cdot LGD\cdot EAD\cdot \Phi\left(\frac{\Phi^{-1}(PD)+\sqrt{\rho}\,\Phi^{-1}(\alpha)}{\sqrt{1-\rho}}\right)$.

In [ ]:
limit = vasicek_var_analytic(0.02, 0.15, 0.45, 1.0, 1000, alpha=0.99)
print(f'Vasicek asymptotic VaR99: {limit:.3f} | theory EL = 1000*0.02*0.45 = 9.0')
for n in [1_000, 5_000, 20_000, 50_000, 100_000]:
    losses = simulate_vasicek_losses(n, 1000, pd=0.02, rho=0.15, lgd=0.45, ead=1.0, seed=42, antithetic=True)
    print(f'N={n:>7,}  VaR99={value_at_risk(losses, 0.99):7.3f}  CVaR99={expected_shortfall(losses, 0.99):7.3f}  EL={losses.mean():.3f} +- {losses.std(ddof=1)/np.sqrt(n):.3f}')

## 2. GBM schemes — Euler (strong order 0.5) vs Milstein (strong order 1.0) vs exact

$E[S_T] = S_0 e^{\mu T} = 105.13$ is known analytically, so the discretisation error is measurable.

In [ ]:
theory = 100*np.exp(0.05)
for steps in [12, 52, 252]:
    errs = {s: simulate_gbm(n_steps=steps, n_paths=20000, seed=7, scheme=s)[:, -1].mean() - theory for s in ['exact','euler','milstein']}
    print(f'steps={steps:>3}  exact={errs["exact"]:+.3f}  euler={errs["euler"]:+.3f}  milstein={errs["milstein"]:+.3f}')

## 3. Kupiec backtest — the test must accept a good model and reject a bad one

In [ ]:
rng = np.random.default_rng(123)
var99 = norm.ppf(0.99)
for name, losses in [('normal (well-specified)', rng.standard_normal(1000)), ('Student-t3 (heavy tails)', rng.standard_t(3, size=1000)/np.sqrt(3))]:
    r = kupiec_test(losses, var99, alpha=0.99)
    print(f'{name}: violations={r["violations"]:.0f}/1000, LR={r["lr_stat"]:.2f}, p={r["p_value"]:.4f}, reject={r["reject_h0_5pct"]}')